In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/DhwaniResearch")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

print("Research folder:", PROJECT_DIR)

In [ ]:
!pip -q install datasets pandas

In [ ]:
from datasets import load_dataset

include_dataset = load_dataset("ai4bharat/INCLUDE")
print(include_dataset)

In [ ]:
import pandas as pd
import re

TARGET_SIGNS = [
    "hello",
    "thankyou",
    "doctor",
    "medicine",
    "cellphone",
    "money",
    "hospital",
    "patient",
    "sick",
    "bathroom",
]

def clean_label(label):
    label = str(label).lower()
    label = re.sub(r"^\d+\.\s*", "", label)
    label = re.sub(r"[^a-z]", "", label)
    return label

all_splits = []

for split_name, split_data in include_dataset.items():
    split_df = split_data.to_pandas()
    split_df["split"] = split_name
    split_df["clean_label"] = split_df["label"].apply(clean_label)
    all_splits.append(split_df)

metadata = pd.concat(all_splits, ignore_index=True)

pilot_metadata = metadata[
    metadata["clean_label"].isin(TARGET_SIGNS)
].copy()

counts = (
    pilot_metadata
    .groupby(["clean_label", "split"])
    .size()
    .unstack(fill_value=0)
)

print(counts)
print("\nTotal videos:", len(pilot_metadata))

In [ ]:
available_signs = set(pilot_metadata["clean_label"].unique())
missing_signs = set(TARGET_SIGNS) - available_signs

print("Available signs:", sorted(available_signs))
print("Missing signs:", sorted(missing_signs))

In [ ]:
output_file = PROJECT_DIR / "dhwani_pilot_metadata.csv"
pilot_metadata.to_csv(output_file, index=False)

print("Saved to:", output_file)

In [ ]:
pilot_full = pilot_metadata[
    pilot_metadata["include_50"] == False
].copy()

print("Total rows:", len(pilot_full))
print("Unique videos:", pilot_full["video_path"].nunique())

print(
    pilot_full
    .groupby(["clean_label", "split"])
    .size()
    .unstack(fill_value=0)
)

In [ ]:
clean_metadata_file = (
    PROJECT_DIR / "dhwani_pilot_include_full.csv"
)

pilot_full.to_csv(clean_metadata_file, index=False)

print("Saved:", clean_metadata_file)

In [ ]:
import requests

zenodo_url = "https://zenodo.org/api/records/4010759"
response = requests.get(zenodo_url)
response.raise_for_status()

zenodo_record = response.json()

for file_info in zenodo_record["files"]:
    size_mb = file_info["size"] / (1024 * 1024)

    print(
        file_info["key"],
        "-",
        round(size_mb, 2),
        "MB"
    )

In [ ]:
!apt-get -qq update
!apt-get -qq install -y aria2

In [ ]:
import subprocess
from pathlib import Path

ZIP_DIR = Path("/content/drive/MyDrive/DhwaniResearch/INCLUDE_zips")
ZIP_DIR.mkdir(parents=True, exist_ok=True)

test_name = "Greetings_1of2.zip"
test_url = (
    "https://zenodo.org/record/4010759/files/"
    f"{test_name}?download=1"
)

subprocess.run([
    "aria2c",
    "-c",
    "-x", "4",
    "-s", "4",
    "--file-allocation=none",
    "--max-tries=20",
    "--retry-wait=10",
    "--timeout=120",
    "--connect-timeout=60",
    "-d", str(ZIP_DIR),
    "-o", test_name,
    test_url
], check=False)

In [ ]:
archive_names = []

for i in range(1, 9):
    archive_names.append(f"Adjectives_{i}of8.zip")

for i in range(1, 3):
    archive_names.append(f"Electronics_{i}of2.zip")
    archive_names.append(f"Greetings_{i}of2.zip")
    archive_names.append(f"Jobs_{i}of2.zip")

for i in range(1, 5):
    archive_names.append(f"Home_{i}of4.zip")
    archive_names.append(f"Places_{i}of4.zip")

for i in range(1, 4):
    archive_names.append(f"Society_{i}of3.zip")

for filename in archive_names:
    file_url = (
        "https://zenodo.org/record/4010759/files/"
        f"{filename}?download=1"
    )

    print("Starting:", filename)

    subprocess.run([
        "aria2c",
        "-c",
        "-x", "4",
        "-s", "4",
        "--file-allocation=none",
        "--max-tries=20",
        "--retry-wait=10",
        "--timeout=120",
        "--connect-timeout=60",
        "-d", str(ZIP_DIR),
        "-o", filename,
        file_url
    ], check=False)

print("Download process finished.")


In [ ]:
downloaded = list(ZIP_DIR.glob("*.zip"))

print("Downloaded archives:", len(downloaded))
print(
    "Downloaded size:",
    round(
        sum(file.stat().st_size for file in downloaded)
        / (1024 ** 3),
        2
    ),
    "GB"
)

In [ ]:
import pandas as pd
import zipfile
import shutil
from pathlib import Path

# Reload metadata if the runtime restarted
if "pilot_full" not in globals():
    pilot_full = pd.read_csv(
        PROJECT_DIR / "dhwani_pilot_include_full.csv"
    )

pilot_full = pilot_full.reset_index(drop=True)

VIDEO_DIR = PROJECT_DIR / "pilot_videos"
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

target_lookup = {}

for row_id, row in pilot_full.iterrows():
    target_path = row["video_path"].replace("\\", "/")

    target_lookup[target_path] = {
        "row_id": row_id,
        "label": row["clean_label"]
    }

local_paths = [None] * len(pilot_full)
found_paths = set()

for zip_path in sorted(ZIP_DIR.glob("*.zip")):
    print("Scanning:", zip_path.name)

    with zipfile.ZipFile(zip_path, "r") as archive:
        for file_info in archive.infolist():

            if file_info.is_dir():
                continue

            archive_path = file_info.filename.replace("\\", "/")
            archive_path = archive_path.lstrip("./")

            matched_target = None

            for target_path in target_lookup:
                if (
                    archive_path == target_path
                    or archive_path.endswith("/" + target_path)
                ):
                    matched_target = target_path
                    break

            if matched_target is None:
                continue

            if matched_target in found_paths:
                continue

            item = target_lookup[matched_target]
            row_id = item["row_id"]
            label = item["label"]

            label_dir = VIDEO_DIR / label
            label_dir.mkdir(parents=True, exist_ok=True)

            original_name = Path(matched_target).name
            output_name = f"{row_id:04d}_{original_name}"
            output_path = label_dir / output_name

            with archive.open(file_info, "r") as source:
                with open(output_path, "wb") as destination:
                    shutil.copyfileobj(
                        source,
                        destination,
                        length=1024 * 1024
                    )

            local_paths[row_id] = str(output_path)
            found_paths.add(matched_target)

pilot_full["local_path"] = local_paths

manifest_path = PROJECT_DIR / "dhwani_pilot_manifest.csv"
pilot_full.to_csv(manifest_path, index=False)

missing = pilot_full[
    pilot_full["local_path"].isna()
]

print("\nExtraction completed.")
print("Extracted videos:", len(found_paths))
print("Missing videos:", len(missing))
print("Manifest:", manifest_path)

In [ ]:
!pip -q install opencv-python-headless


In [ ]:
import cv2
import pandas as pd
from pathlib import Path

manifest_path = PROJECT_DIR / "dhwani_pilot_manifest.csv"
manifest = pd.read_csv(manifest_path)

quality_rows = []

for _, row in manifest.iterrows():
    video_path = Path(row["local_path"])

    result = {
        "video_path": str(video_path),
        "label": row["clean_label"],
        "split": row["split"],
        "exists": video_path.exists(),
        "file_size_mb": 0,
        "readable": False,
        "fps": 0,
        "frame_count": 0,
        "duration_seconds": 0,
    }

    if video_path.exists():
        result["file_size_mb"] = round(
            video_path.stat().st_size / (1024 * 1024),
            3
        )

        capture = cv2.VideoCapture(str(video_path))

        if capture.isOpened():
            fps = capture.get(cv2.CAP_PROP_FPS)
            frame_count = int(
                capture.get(cv2.CAP_PROP_FRAME_COUNT)
            )

            success, frame = capture.read()

            result["readable"] = (
                success
                and frame is not None
                and frame_count > 0
            )
            result["fps"] = round(fps, 2)
            result["frame_count"] = frame_count

            if fps > 0:
                result["duration_seconds"] = round(
                    frame_count / fps,
                    2
                )

        capture.release()

    quality_rows.append(result)

quality_report = pd.DataFrame(quality_rows)

quality_path = PROJECT_DIR / "dhwani_video_quality.csv"
quality_report.to_csv(quality_path, index=False)

print("Total videos:", len(quality_report))
print("Missing files:", (~quality_report["exists"]).sum())
print("Unreadable videos:", (~quality_report["readable"]).sum())
print("Zero-size files:", (quality_report["file_size_mb"] == 0).sum())

print("\nVideos by class:")
print(quality_report["label"].value_counts().sort_index())

print("\nDuration summary:")
print(
    quality_report["duration_seconds"].describe()
)

print("\nSaved report:")
print(quality_path)

In [ ]:
from IPython.display import display, Video, Markdown

for label in sorted(quality_report["label"].unique()):
    sample = (
        quality_report[
            quality_report["label"] == label
        ]
        .sample(1, random_state=42)
        .iloc[0]
    )

    display(Markdown(f"### {label.upper()}"))
    display(
        Video(
            sample["video_path"],
            embed=True,
            width=400
        )
    )

In [ ]:
import cv2
import matplotlib.pyplot as plt

labels = sorted(quality_report["label"].unique())

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

for ax, label in zip(axes, labels):
    sample = (
        quality_report[quality_report["label"] == label]
        .sample(1, random_state=42)
        .iloc[0]
    )

    cap = cv2.VideoCapture(sample["video_path"])
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Read a middle frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, max(frame_count // 2, 0))
    success, frame = cap.read()
    cap.release()

    if success:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        ax.imshow(frame)
        ax.set_title(label.upper())
    else:
        ax.set_title(label.upper() + " - FAILED")

    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
%cd /content

!git clone https://github.com/AI4Bharat/INCLUDE.git
!pip install -q -r /content/INCLUDE/requirements.txt

In [ ]:
import pandas as pd
from pathlib import Path

manifest_path = "/content/drive/MyDrive/DhwaniResearch/dhwani_pilot_manifest.csv"

manifest = pd.read_csv(manifest_path)

print(manifest.columns.tolist())
print(manifest.groupby("split").size())


In [ ]:
import sys

sys.path.insert(0, "/content/INCLUDE")

from generate_keypoints import process_video
from tqdm.auto import tqdm

In [ ]:
!pip install -q mediapipe==1.0.1

In [ ]:
import mediapipe as mp

print(mp.__version__)
print(hasattr(mp, "tasks"))

In [ ]:
!wget -q -O /content/hand_landmarker.task \
https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

!wget -q -O /content/pose_landmarker.task \
https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task

In [ ]:
import cv2
import json
import numpy as np
import mediapipe as mp

from pathlib import Path
from mediapipe.tasks import python
from mediapipe.tasks.python import vision


HAND_MODEL = "/content/hand_landmarker.task"
POSE_MODEL = "/content/pose_landmarker.task"


def padded_xy(landmarks, count):
    xs = []
    ys = []

    for landmark in landmarks[:count]:
        xs.append(float(landmark.x))
        ys.append(float(landmark.y))

    while len(xs) < count:
        xs.append(float("nan"))
        ys.append(float("nan"))

    return xs, ys


def extract_one_video(video_path, save_dir):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    label = Path(video_path).parent.name.lower()
    uid = f"{label}_{Path(video_path).stem}"

    hand_options = vision.HandLandmarkerOptions(
        base_options=python.BaseOptions(
            model_asset_path=HAND_MODEL
        ),
        running_mode=vision.RunningMode.VIDEO,
        num_hands=2,
        min_hand_detection_confidence=0.5,
        min_hand_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )

    pose_options = vision.PoseLandmarkerOptions(
        base_options=python.BaseOptions(
            model_asset_path=POSE_MODEL
        ),
        running_mode=vision.RunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )

    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps <= 0:
        fps = 30.0

    pose_x_all = []
    pose_y_all = []
    hand1_x_all = []
    hand1_y_all = []
    hand2_x_all = []
    hand2_y_all = []

    frame_index = 0

    with (
        vision.HandLandmarker.create_from_options(hand_options) as hand_detector,
        vision.PoseLandmarker.create_from_options(pose_options) as pose_detector,
    ):
        while True:
            success, frame = cap.read()

            if not success:
                break

            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=rgb_frame,
            )

            timestamp_ms = int(frame_index * 1000 / fps)

            hand_result = hand_detector.detect_for_video(
                image,
                timestamp_ms,
            )

            pose_result = pose_detector.detect_for_video(
                image,
                timestamp_ms,
            )

            # Pose: keep first 25 landmarks to match INCLUDE format
            if pose_result.pose_landmarks:
                pose_landmarks = pose_result.pose_landmarks[0]
            else:
                pose_landmarks = []

            pose_x, pose_y = padded_xy(pose_landmarks, 25)

            # Two hands: 21 landmarks each
            hand1_x = [float("nan")] * 21
            hand1_y = [float("nan")] * 21
            hand2_x = [float("nan")] * 21
            hand2_y = [float("nan")] * 21

            for i, hand_landmarks in enumerate(hand_result.hand_landmarks):
                if i >= len(hand_result.handedness):
                    continue

                handedness = hand_result.handedness[i][0]
                side = handedness.category_name.lower()

                hx, hy = padded_xy(hand_landmarks, 21)

                if side == "left":
                    hand1_x, hand1_y = hx, hy
                elif side == "right":
                    hand2_x, hand2_y = hx, hy

            pose_x_all.append(pose_x)
            pose_y_all.append(pose_y)
            hand1_x_all.append(hand1_x)
            hand1_y_all.append(hand1_y)
            hand2_x_all.append(hand2_x)
            hand2_y_all.append(hand2_y)

            frame_index += 1

    cap.release()

    output = {
        "uid": uid,
        "label": label,
        "pose_x": pose_x_all,
        "pose_y": pose_y_all,
        "hand1_x": hand1_x_all,
        "hand1_y": hand1_y_all,
        "hand2_x": hand2_x_all,
        "hand2_y": hand2_y_all,
        "n_frames": frame_index,
    }

    output_file = save_dir / f"{uid}.json"

    with open(output_file, "w") as f:
        json.dump(output, f)

    return output_file

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import pandas as pd

manifest_path = (
    "/content/drive/MyDrive/"
    "DhwaniResearch/dhwani_pilot_manifest.csv"
)

manifest = pd.read_csv(manifest_path)

print(manifest.columns.tolist())
print(manifest.shape)
print(manifest.head())

In [ ]:
import json
from pathlib import Path

sample_video = manifest.iloc[0]["local_path"]

print("Exists:", Path(sample_video).exists())
print("Path:", sample_video)

test_output = output_root / "test_local"
test_output.mkdir(parents=True, exist_ok=True)

created_file = extract_one_video(
    sample_video,
    test_output
)

print("Created:", created_file)

In [ ]:
with open(created_file, "r") as f:
    test_data = json.load(f)

print("Label:", test_data["label"])
print("Frames:", test_data["n_frames"])
print("Pose frames:", len(test_data["pose_x"]))

In [ ]:
from tqdm.auto import tqdm

for split in ["train", "val", "test"]:
    split_output = output_root / f"pilot_{split}_keypoints"
    split_output.mkdir(parents=True, exist_ok=True)

    split_data = manifest[manifest["split"] == split]

    print(f"Processing {split}: {len(split_data)} videos")

    for _, row in tqdm(
        split_data.iterrows(),
        total=len(split_data),
        desc=f"{split} keypoints"
    ):
        extract_one_video(
            row["local_path"],
            split_output
        )

print("All videos processed.")

In [ ]:
import json

for split in ["train", "val", "test"]:
    folder = output_root / f"pilot_{split}_keypoints"
    files = list(folder.glob("*.json"))

    valid = 0
    empty = 0

    for file in files:
        with open(file, "r") as f:
            data = json.load(f)

        if data["n_frames"] > 0:
            valid += 1
        else:
            empty += 1

    print(
        split,
        "files:", len(files),
        "valid:", valid,
        "empty:", empty
    )

In [ ]:
import json

labels = sorted(manifest["clean_label"].unique())

label_to_id = {
    label: index
    for index, label in enumerate(labels)
}

id_to_label = {
    index: label
    for label, index in label_to_id.items()
}

print(label_to_id)

label_map_path = (
    "/content/drive/MyDrive/"
    "DhwaniResearch/pilot_label_map.json"
)

with open(label_map_path, "w") as f:
    json.dump(
        {
            "label_to_id": label_to_id,
            "id_to_label": id_to_label
        },
        f,
        indent=2
    )

print("Saved:", label_map_path)

In [ ]:
import json
import numpy as np
import torch

from pathlib import Path
from torch.utils.data import Dataset, DataLoader


MAX_FRAMES = 169
FRAME_WIDTH = 1920
FRAME_HEIGHT = 1080


with open(label_map_path, "r") as f:
    label_maps = json.load(f)

label_to_id = label_maps["label_to_id"]


class PilotKeypointsDataset(Dataset):
    def __init__(self, folder, label_to_id, max_frames=169):
        self.folder = Path(folder)
        self.files = sorted(self.folder.glob("*.json"))
        self.label_to_id = label_to_id
        self.max_frames = max_frames

    def __len__(self):
        return len(self.files)

    def interpolate_coordinates(self, values):
        values = np.asarray(values, dtype=np.float32)

        for landmark_index in range(values.shape[1]):
            for coordinate_index in range(2):
                series = values[:, landmark_index, coordinate_index]
                valid = ~np.isnan(series)

                if valid.any():
                    indices = np.arange(len(series))
                    series[~valid] = np.interp(
                        indices[~valid],
                        indices[valid],
                        series[valid]
                    )
                else:
                    series[:] = 0.0

                values[:, landmark_index, coordinate_index] = series

        return values

    def __getitem__(self, index):
        file_path = self.files[index]

        with open(file_path, "r") as f:
            item = json.load(f)

        pose = np.stack(
            [item["pose_x"], item["pose_y"]],
            axis=-1
        )

        hand1 = np.stack(
            [item["hand1_x"], item["hand1_y"]],
            axis=-1
        )

        hand2 = np.stack(
            [item["hand2_x"], item["hand2_y"]],
            axis=-1
        )

        pose = self.interpolate_coordinates(pose)
        hand1 = self.interpolate_coordinates(hand1)
        hand2 = self.interpolate_coordinates(hand2)

        # Scale coordinates like the original INCLUDE preprocessing
        pose[:, :, 0] *= FRAME_WIDTH
        pose[:, :, 1] *= FRAME_HEIGHT

        hand1[:, :, 0] *= FRAME_WIDTH
        hand1[:, :, 1] *= FRAME_HEIGHT

        hand2[:, :, 0] *= FRAME_WIDTH
        hand2[:, :, 1] *= FRAME_HEIGHT

        features = np.concatenate(
            [pose, hand1, hand2],
            axis=1
        )

        features = features.reshape(features.shape[0], -1)

        # Limit or pad to 169 frames
        if features.shape[0] > self.max_frames:
            features = features[:self.max_frames]

        elif features.shape[0] < self.max_frames:
            padding = np.zeros(
                (
                    self.max_frames - features.shape[0],
                    features.shape[1]
                ),
                dtype=np.float32
            )

            features = np.concatenate(
                [features, padding],
                axis=0
            )

        label_name = item["label"]
        label_id = self.label_to_id[label_name]

        return {
            "data": torch.tensor(features, dtype=torch.float32),
            "label": torch.tensor(label_id, dtype=torch.long),
            "uid": item["uid"],
            "label_name": label_name
        }

In [ ]:
train_dir = output_root / "pilot_train_keypoints"
val_dir = output_root / "pilot_val_keypoints"
test_dir = output_root / "pilot_test_keypoints"

train_dataset = PilotKeypointsDataset(
    train_dir,
    label_to_id,
    MAX_FRAMES
)

val_dataset = PilotKeypointsDataset(
    val_dir,
    label_to_id,
    MAX_FRAMES
)

test_dataset = PilotKeypointsDataset(
    test_dir,
    label_to_id,
    MAX_FRAMES
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

In [ ]:
batch = next(iter(train_loader))

print("Data shape:", batch["data"].shape)
print("Label shape:", batch["label"].shape)
print("Labels:", batch["label"])
print("First UID:", batch["uid"][0])

In [ ]:
import sys
import json
import torch

sys.path.insert(0, "/content/INCLUDE")

from configs import TransformerConfig
from models import Transformer

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


In [ ]:
with open("/content/INCLUDE/pretrained_links.json", "r") as f:
    pretrained_links = json.load(f)

print(pretrained_links.keys())

In [ ]:
from pathlib import Path

In [ ]:
checkpoint_name = "include_no_cnn_transformer_small.pth"
checkpoint_path = f"/content/{checkpoint_name}"

if not Path(checkpoint_path).exists():
    torch.hub.download_url_to_file(
        pretrained_links[checkpoint_name],
        checkpoint_path,
        progress=True
    )

print("Checkpoint ready:", checkpoint_path)

In [ ]:
import torch
import torch.nn.functional as F

from models import Transformer as OriginalTransformer


class CompatibleTransformer(OriginalTransformer):
    def forward(self, x):
        x = self.l1(x)
        x = self.embedding(x)

        for layer in self.layers:
            result = layer(x)

            # Old Transformers: tuple
            # New Transformers: tensor
            if isinstance(result, (tuple, list)):
                x = result[0]
            else:
                x = result

        x = torch.max(x, dim=1).values
        x = F.dropout(
            x,
            p=0.2,
            training=self.training
        )

        return self.l2(x)

In [ ]:
config = TransformerConfig(
    size="small",
    max_position_embeddings=256
)

model = CompatibleTransformer(
    config=config,
    n_classes=10
).to(device)

In [ ]:
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

pretrained_state = checkpoint["model"]
current_state = model.state_dict()

compatible_state = {
    key: value
    for key, value in pretrained_state.items()
    if key in current_state
    and current_state[key].shape == value.shape
}

missing_keys, unexpected_keys = model.load_state_dict(
    compatible_state,
    strict=False
)

print("Loaded weights:", len(compatible_state))
print("Missing keys:", missing_keys)

In [ ]:
batch = next(iter(train_loader))
input_data = batch["data"].to(device)

model.eval()

with torch.no_grad():
    output = model(input_data)

print("Model output shape:", output.shape)

In [ ]:
import torch
import torch.nn as nn

# Freeze the pretrained feature extractor
for parameter in model.parameters():
    parameter.requires_grad = False

# Train only the new classification layer
for parameter in model.l2.parameters():
    parameter.requires_grad = True

trainable_parameters = [
    parameter
    for parameter in model.parameters()
    if parameter.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=1e-3,
    weight_decay=1e-4
)

criterion = nn.CrossEntropyLoss()

best_val_accuracy = 0.0
best_model_path = (
    "/content/drive/MyDrive/"
    "DhwaniResearch/pilot_head_best.pth"
)

In [ ]:
def run_epoch(loader, training=True):
    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for batch in loader:
        inputs = batch["data"].to(device)
        labels = batch["label"].to(device)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            logits = model(inputs)
            loss = criterion(logits, labels)

            if training:
                loss.backward()
                optimizer.step()

        predictions = torch.argmax(logits, dim=1)

        total_loss += loss.item() * labels.size(0)
        total_correct += (predictions == labels).sum().item()
        total_samples += labels.size(0)

    average_loss = total_loss / total_samples
    accuracy = total_correct / total_samples

    return average_loss, accuracy

In [ ]:
for epoch in range(10):
    train_loss, train_accuracy = run_epoch(
        train_loader,
        training=True
    )

    val_loss, val_accuracy = run_epoch(
        val_loader,
        training=False
    )

    print(
        f"Epoch {epoch + 1}/10 | "
        f"Train loss: {train_loss:.4f} | "
        f"Train accuracy: {train_accuracy:.3f} | "
        f"Validation loss: {val_loss:.4f} | "
        f"Validation accuracy: {val_accuracy:.3f}"
    )

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy

        torch.save(
            {
                "model": model.state_dict(),
                "val_accuracy": val_accuracy,
                "epoch": epoch + 1
            },
            best_model_path
        )

        print("Saved best model.")

In [ ]:
best_checkpoint = torch.load(
    best_model_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(best_checkpoint["model"])

In [ ]:
for parameter in model.parameters():
    parameter.requires_grad = True

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-5,
    weight_decay=1e-4
)

criterion = nn.CrossEntropyLoss()

print("Full model is trainable.")

In [ ]:
best_val_accuracy = 0.0
patience = 5
epochs_without_improvement = 0

full_model_path = (
    "/content/drive/MyDrive/"
    "DhwaniResearch/pilot_include_transformer_best.pth"
)

for epoch in range(20):
    train_loss, train_accuracy = run_epoch(
        train_loader,
        training=True
    )

    val_loss, val_accuracy = run_epoch(
        val_loader,
        training=False
    )

    print(
        f"Fine-tune epoch {epoch + 1}/20 | "
        f"Train loss: {train_loss:.4f} | "
        f"Train accuracy: {train_accuracy:.3f} | "
        f"Validation loss: {val_loss:.4f} | "
        f"Validation accuracy: {val_accuracy:.3f}"
    )

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        epochs_without_improvement = 0

        torch.save(
            {
                "model": model.state_dict(),
                "val_accuracy": val_accuracy,
                "epoch": epoch + 1
            },
            full_model_path
        )

        print("Saved best full model.")

    else:
        epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print("Early stopping.")
            break

In [ ]:
best_checkpoint = torch.load(
    full_model_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(best_checkpoint["model"])
model.eval()

In [ ]:
# COMPLETE TEST EVALUATION CELL

from google.colab import drive
drive.mount("/content/drive")

import json
import torch
import transformers
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)


# --------------------------------------------------
# 1. Paths
# --------------------------------------------------

research_dir = Path(
    "/content/drive/MyDrive/DhwaniResearch"
)

label_map_path = research_dir / "pilot_label_map.json"

checkpoint_path = (
    research_dir /
    "pilot_include_transformer_best.pth"
)

test_keypoints_dir = (
    research_dir /
    "pilot_keypoints/pilot_test_keypoints"
)

assert research_dir.exists(), (
    f"Research folder not found: {research_dir}"
)

assert label_map_path.exists(), (
    f"Label map not found: {label_map_path}"
)

assert checkpoint_path.exists(), (
    f"Checkpoint not found: {checkpoint_path}"
)

assert test_keypoints_dir.exists(), (
    f"Test keypoints not found: {test_keypoints_dir}"
)

print("All required files found.")


# --------------------------------------------------
# 2. Load label map
# --------------------------------------------------

with open(label_map_path, "r") as file:
    label_maps = json.load(file)

label_to_id = {
    label: int(index)
    for label, index
    in label_maps["label_to_id"].items()
}

class_names = [
    label
    for label, index in sorted(
        label_to_id.items(),
        key=lambda item: item[1]
    )
]

print("Classes:", class_names)


# --------------------------------------------------
# 3. Recreate test dataset
# --------------------------------------------------

MAX_FRAMES = 169
FRAME_WIDTH = 1920
FRAME_HEIGHT = 1080


class PilotKeypointsDataset(Dataset):

    def __init__(
        self,
        folder,
        label_to_id,
        max_frames=169
    ):
        self.folder = Path(folder)

        self.files = sorted(
            self.folder.glob("*.json")
        )

        self.label_to_id = label_to_id
        self.max_frames = max_frames

    def __len__(self):
        return len(self.files)

    def interpolate_coordinates(self, values):
        values = np.asarray(
            values,
            dtype=np.float32
        )

        for landmark_index in range(
            values.shape[1]
        ):
            for coordinate_index in range(2):

                series = values[
                    :,
                    landmark_index,
                    coordinate_index
                ]

                valid = np.isfinite(series)

                if valid.any():
                    indices = np.arange(
                        len(series)
                    )

                    series[~valid] = np.interp(
                        indices[~valid],
                        indices[valid],
                        series[valid]
                    )
                else:
                    series[:] = 0.0

                values[
                    :,
                    landmark_index,
                    coordinate_index
                ] = series

        return values

    def __getitem__(self, index):
        file_path = self.files[index]

        with open(file_path, "r") as file:
            item = json.load(file)

        pose = np.stack(
            [
                item["pose_x"],
                item["pose_y"]
            ],
            axis=-1
        )

        hand1 = np.stack(
            [
                item["hand1_x"],
                item["hand1_y"]
            ],
            axis=-1
        )

        hand2 = np.stack(
            [
                item["hand2_x"],
                item["hand2_y"]
            ],
            axis=-1
        )

        pose = self.interpolate_coordinates(pose)
        hand1 = self.interpolate_coordinates(hand1)
        hand2 = self.interpolate_coordinates(hand2)

        pose[:, :, 0] *= FRAME_WIDTH
        pose[:, :, 1] *= FRAME_HEIGHT

        hand1[:, :, 0] *= FRAME_WIDTH
        hand1[:, :, 1] *= FRAME_HEIGHT

        hand2[:, :, 0] *= FRAME_WIDTH
        hand2[:, :, 1] *= FRAME_HEIGHT

        features = np.concatenate(
            [pose, hand1, hand2],
            axis=1
        )

        features = features.reshape(
            features.shape[0],
            -1
        )

        if features.shape[0] > self.max_frames:
            features = features[:self.max_frames]

        elif features.shape[0] < self.max_frames:

            padding = np.zeros(
                (
                    self.max_frames
                    - features.shape[0],
                    features.shape[1]
                ),
                dtype=np.float32
            )

            features = np.concatenate(
                [features, padding],
                axis=0
            )

        label_name = item["label"]
        label_id = self.label_to_id[label_name]

        return {
            "data": torch.tensor(
                features,
                dtype=torch.float32
            ),
            "label": torch.tensor(
                label_id,
                dtype=torch.long
            ),
            "uid": item["uid"]
        }


test_dataset = PilotKeypointsDataset(
    test_keypoints_dir,
    label_to_id,
    MAX_FRAMES
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

print("Test videos loaded:", len(test_dataset))


# --------------------------------------------------
# 4. Recreate INCLUDE Transformer
# --------------------------------------------------

class TransformerConfig:

    def __init__(
        self,
        size="small",
        max_position_embeddings=256
    ):
        self.size = size
        self.input_size = 134
        self.max_position_embeddings = (
            max_position_embeddings
        )

        self.layer_norm_eps = 1e-12
        self.hidden_dropout_prob = 0.1

        if size == "small":
            self.hidden_size = 256
            self.num_attention_heads = 4
            self.num_hidden_layers = 2
        else:
            self.hidden_size = 512
            self.num_attention_heads = 8
            self.num_hidden_layers = 4

        self.model_config = (
            transformers.BertConfig(
                hidden_size=self.hidden_size,
                num_attention_heads=(
                    self.num_attention_heads
                ),
                num_hidden_layers=(
                    self.num_hidden_layers
                ),
                max_position_embeddings=(
                    self.max_position_embeddings
                )
            )
        )


class PositionEmbedding(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.position_embeddings = nn.Embedding(
            config.max_position_embeddings,
            config.hidden_size
        )

        self.LayerNorm = nn.LayerNorm(
            config.hidden_size,
            eps=config.layer_norm_eps
        )

        self.dropout = nn.Dropout(
            config.hidden_dropout_prob
        )

        self.register_buffer(
            "position_ids",
            torch.arange(
                config.max_position_embeddings
            ).expand((1, -1))
        )

    def forward(self, x):
        sequence_length = x.size(1)

        position_ids = self.position_ids[
            :,
            :sequence_length
        ]

        position_embeddings = (
            self.position_embeddings(
                position_ids
            )
        )

        embeddings = (
            x + position_embeddings
        )

        embeddings = self.LayerNorm(
            embeddings
        )

        embeddings = self.dropout(
            embeddings
        )

        return embeddings


class CompatibleTransformer(nn.Module):

    def __init__(
        self,
        config,
        number_of_classes
    ):
        super().__init__()

        self.l1 = nn.Linear(
            config.input_size,
            config.hidden_size
        )

        self.embedding = PositionEmbedding(
            config
        )

        self.layers = nn.ModuleList([
            transformers.BertLayer(
                config.model_config
            )
            for _ in range(
                config.num_hidden_layers
            )
        ])

        self.l2 = nn.Linear(
            config.hidden_size,
            number_of_classes
        )

    def forward(self, x):
        x = self.l1(x)
        x = self.embedding(x)

        for layer in self.layers:
            result = layer(x)

            if isinstance(
                result,
                (tuple, list)
            ):
                x = result[0]
            else:
                x = result

        x = torch.max(
            x,
            dim=1
        ).values

        x = F.dropout(
            x,
            p=0.2,
            training=self.training
        )

        return self.l2(x)


# --------------------------------------------------
# 5. Load trained model
# --------------------------------------------------

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

config = TransformerConfig(
    size="small",
    max_position_embeddings=256
)

model = CompatibleTransformer(
    config=config,
    number_of_classes=len(class_names)
).to(device)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model"],
    strict=True
)

model.eval()

print("Model loaded on:", device)


# --------------------------------------------------
# 6. Run predictions
# --------------------------------------------------

all_true = []
all_pred = []
all_uids = []

with torch.no_grad():

    for batch in test_loader:

        inputs = batch["data"].to(device)
        labels = batch["label"].to(device)

        logits = model(inputs)

        predictions = torch.argmax(
            logits,
            dim=1
        )

        all_true.extend(
            labels.cpu().tolist()
        )

        all_pred.extend(
            predictions.cpu().tolist()
        )

        all_uids.extend(
            batch["uid"]
        )


# --------------------------------------------------
# 7. Calculate metrics
# --------------------------------------------------

accuracy = accuracy_score(
    all_true,
    all_pred
)

macro_precision = precision_score(
    all_true,
    all_pred,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    all_true,
    all_pred,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    all_true,
    all_pred,
    average="macro",
    zero_division=0
)

correct_count = sum(
    true == predicted
    for true, predicted
    in zip(all_true, all_pred)
)

print("\nFINAL TEST RESULTS")
print("------------------")
print("Test videos:", len(all_true))
print("Correct predictions:", correct_count)
print("Test accuracy:", round(accuracy, 4))
print(
    "Macro precision:",
    round(macro_precision, 4)
)
print(
    "Macro recall:",
    round(macro_recall, 4)
)
print(
    "Macro F1:",
    round(macro_f1, 4)
)

print("\nCLASSIFICATION REPORT\n")

print(
    classification_report(
        all_true,
        all_pred,
        labels=list(
            range(len(class_names))
        ),
        target_names=class_names,
        zero_division=0
    )
)


# --------------------------------------------------
# 8. Save predictions
# --------------------------------------------------

prediction_table = pd.DataFrame({
    "uid": all_uids,

    "true_label": [
        class_names[index]
        for index in all_true
    ],

    "predicted_label": [
        class_names[index]
        for index in all_pred
    ],

    "correct": [
        true == predicted
        for true, predicted
        in zip(all_true, all_pred)
    ]
})

prediction_path = (
    research_dir /
    "pilot_test_predictions.csv"
)

prediction_table.to_csv(
    prediction_path,
    index=False
)

print(
    "\nPredictions saved:",
    prediction_path
)


# --------------------------------------------------
# 9. Save confusion matrix
# --------------------------------------------------

confusion_path = (
    research_dir /
    "pilot_confusion_matrix.png"
)

matrix = confusion_matrix(
    all_true,
    all_pred,
    labels=list(
        range(len(class_names))
    )
)

fig, ax = plt.subplots(
    figsize=(9, 8)
)

display = ConfusionMatrixDisplay(
    confusion_matrix=matrix,
    display_labels=class_names
)

display.plot(
    ax=ax,
    cmap="Blues",
    xticks_rotation=45,
    colorbar=False
)

ax.set_title(
    "INCLUDE Transformer: "
    "10-Class Test Results"
)

fig.tight_layout()

fig.savefig(
    confusion_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "Confusion matrix saved:",
    confusion_path
)

In [ ]:
import json
import pandas as pd
from sklearn.metrics import classification_report

results = {
    "model": "Fine-tuned AI4Bharat INCLUDE Transformer Small",
    "classes": 10,
    "train_videos": 120,
    "validation_videos": 13,
    "test_videos": 34,
    "correct_predictions": 32,
    "test_accuracy": float(accuracy),
    "macro_precision": float(macro_precision),
    "macro_recall": float(macro_recall),
    "macro_f1": float(macro_f1),
    "accuracy_ci_95_wilson": [
        0.809,
        0.984
    ]
}

results_path = (
    research_dir /
    "pilot_final_metrics.json"
)

with open(results_path, "w") as file:
    json.dump(results, file, indent=2)

report = classification_report(
    all_true,
    all_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

report_path = (
    research_dir /
    "pilot_classification_report.csv"
)

pd.DataFrame(report).transpose().to_csv(
    report_path
)

print("Metrics saved:", results_path)
print("Report saved:", report_path)

print("\nIncorrect predictions:")
display(
    prediction_table[
        prediction_table["correct"] == False
    ]
)

In [ ]:
from IPython.display import display as show_table

incorrect_predictions = prediction_table[
    prediction_table["correct"] == False
]

print("Incorrect predictions:", len(incorrect_predictions))

show_table(incorrect_predictions)


In [ ]:
import json
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

head_checkpoint_path = (
    research_dir /
    "pilot_head_best.pth"
)

assert head_checkpoint_path.exists(), (
    f"Checkpoint missing: {head_checkpoint_path}"
)

head_checkpoint = torch.load(
    head_checkpoint_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    head_checkpoint["model"],
    strict=True
)

model.eval()

head_true = []
head_pred = []

with torch.no_grad():
    for batch in test_loader:
        inputs = batch["data"].to(device)
        labels = batch["label"].to(device)

        logits = model(inputs)

        predictions = torch.argmax(
            logits,
            dim=1
        )

        head_true.extend(
            labels.cpu().tolist()
        )

        head_pred.extend(
            predictions.cpu().tolist()
        )

head_accuracy = accuracy_score(
    head_true,
    head_pred
)

head_precision = precision_score(
    head_true,
    head_pred,
    average="macro",
    zero_division=0
)

head_recall = recall_score(
    head_true,
    head_pred,
    average="macro",
    zero_division=0
)

head_f1 = f1_score(
    head_true,
    head_pred,
    average="macro",
    zero_division=0
)

head_correct = sum(
    true == predicted
    for true, predicted
    in zip(head_true, head_pred)
)

print("HEAD-ONLY TEST RESULTS")
print("----------------------")
print("Correct:", head_correct, "/", len(head_true))
print("Accuracy:", round(head_accuracy, 4))
print("Macro precision:", round(head_precision, 4))
print("Macro recall:", round(head_recall, 4))
print("Macro F1:", round(head_f1, 4))

head_results = {
    "model": "Pretrained INCLUDE Transformer with frozen backbone",
    "correct_predictions": head_correct,
    "test_videos": len(head_true),
    "test_accuracy": float(head_accuracy),
    "macro_precision": float(head_precision),
    "macro_recall": float(head_recall),
    "macro_f1": float(head_f1)
}

head_results_path = (
    research_dir /
    "pilot_head_only_metrics.json"
)

with open(head_results_path, "w") as file:
    json.dump(
        head_results,
        file,
        indent=2
    )

print("Saved:", head_results_path)

In [ ]:
for epoch in range(30, 50):

    train_loss, train_accuracy = run_scratch_epoch(
        scratch_model,
        train_loader,
        scratch_optimizer
    )

    val_loss, val_accuracy = run_scratch_epoch(
        scratch_model,
        val_loader
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy
    })

    print(
        f"Scratch epoch {epoch + 1}/50 | "
        f"Train loss: {train_loss:.4f} | "
        f"Train accuracy: {train_accuracy:.3f} | "
        f"Validation loss: {val_loss:.4f} | "
        f"Validation accuracy: {val_accuracy:.3f}"
    )

    improved = (
        val_accuracy > best_val_accuracy
        or (
            val_accuracy == best_val_accuracy
            and val_loss < best_val_loss
        )
    )

    if improved:
        best_val_accuracy = val_accuracy
        best_val_loss = val_loss
        epochs_without_improvement = 0

        torch.save(
            {
                "model": scratch_model.state_dict(),
                "epoch": epoch + 1,
                "val_accuracy": val_accuracy,
                "val_loss": val_loss,
                "seed": SEED
            },
            scratch_checkpoint_path
        )

        print("Saved improved scratch model.")

    else:
        epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print("Early stopping.")
            break

pd.DataFrame(history).to_csv(
    history_path,
    index=False
)

print("Updated history:", history_path)

In [ ]:
scratch_checkpoint = torch.load(
    scratch_checkpoint_path,
    map_location=device,
    weights_only=False
)

scratch_model.load_state_dict(
    scratch_checkpoint["model"]
)

scratch_model.eval()

scratch_true = []
scratch_pred = []

with torch.no_grad():
    for batch in test_loader:
        inputs = batch["data"].to(device)
        labels = batch["label"].to(device)

        logits = scratch_model(inputs)

        predictions = torch.argmax(
            logits,
            dim=1
        )

        scratch_true.extend(
            labels.cpu().tolist()
        )

        scratch_pred.extend(
            predictions.cpu().tolist()
        )

scratch_accuracy = accuracy_score(
    scratch_true,
    scratch_pred
)

scratch_precision = precision_score(
    scratch_true,
    scratch_pred,
    average="macro",
    zero_division=0
)

scratch_recall = recall_score(
    scratch_true,
    scratch_pred,
    average="macro",
    zero_division=0
)

scratch_f1 = f1_score(
    scratch_true,
    scratch_pred,
    average="macro",
    zero_division=0
)

scratch_correct = sum(
    true == predicted
    for true, predicted
    in zip(scratch_true, scratch_pred)
)

print("\nSCRATCH MODEL TEST RESULTS")
print("--------------------------")
print(
    "Correct:",
    scratch_correct,
    "/",
    len(scratch_true)
)
print("Accuracy:", round(scratch_accuracy, 4))
print(
    "Macro precision:",
    round(scratch_precision, 4)
)
print(
    "Macro recall:",
    round(scratch_recall, 4)
)
print("Macro F1:", round(scratch_f1, 4))

In [ ]:
scratch_checkpoint = torch.load(
    scratch_checkpoint_path,
    map_location=device,
    weights_only=False
)

scratch_model.load_state_dict(
    scratch_checkpoint["model"]
)

scratch_model.eval()

scratch_true = []
scratch_pred = []

with torch.no_grad():
    for batch in test_loader:
        inputs = batch["data"].to(device)
        labels = batch["label"].to(device)

        predictions = torch.argmax(
            scratch_model(inputs),
            dim=1
        )

        scratch_true.extend(labels.cpu().tolist())
        scratch_pred.extend(predictions.cpu().tolist())

scratch_accuracy = accuracy_score(
    scratch_true,
    scratch_pred
)

scratch_precision = precision_score(
    scratch_true,
    scratch_pred,
    average="macro",
    zero_division=0
)

scratch_recall = recall_score(
    scratch_true,
    scratch_pred,
    average="macro",
    zero_division=0
)

scratch_f1 = f1_score(
    scratch_true,
    scratch_pred,
    average="macro",
    zero_division=0
)

scratch_correct = sum(
    true == predicted
    for true, predicted
    in zip(scratch_true, scratch_pred)
)

print("FINAL SCRATCH RESULTS")
print("Best epoch:", scratch_checkpoint["epoch"])
print("Correct:", scratch_correct, "/", len(scratch_true))
print("Accuracy:", round(scratch_accuracy, 4))
print("Macro precision:", round(scratch_precision, 4))
print("Macro recall:", round(scratch_recall, 4))
print("Macro F1:", round(scratch_f1, 4))

In [ ]:
from IPython.display import display as show_table
import matplotlib.pyplot as plt

show_table(
    comparison.round(4)
)

chart_data = comparison.set_index("Model")[
    ["Accuracy", "Macro F1"]
]

ax = chart_data.plot(
    kind="bar",
    figsize=(10, 6),
    color=["#333333", "#999999"]
)

ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_xlabel("")
ax.set_title(
    "Comparison of Pilot Sign Recognition Models"
)

ax.legend([
    "Accuracy",
    "Macro-F1"
])

ax.grid(
    axis="y",
    alpha=0.25
)

plt.xticks(
    rotation=15,
    ha="right"
)

plt.tight_layout()

comparison_chart_path = (
    research_dir /
    "pilot_model_comparison.png"
)

plt.savefig(
    comparison_chart_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Scratch metrics:", scratch_metrics_path)
print("Comparison table:", comparison_path)
print("Comparison chart:", comparison_chart_path)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json
import torch
import transformers
import numpy as np
import torch.nn as nn
import torch.nn.functional as F

from pathlib import Path
from torch.utils.data import Dataset, DataLoader


# --------------------------------------------------
# Paths and labels
# --------------------------------------------------

research_dir = Path(
    "/content/drive/MyDrive/DhwaniResearch"
)

label_map_path = (
    research_dir /
    "pilot_label_map.json"
)

with open(label_map_path, "r") as file:
    label_maps = json.load(file)

label_to_id = {
    label: int(index)
    for label, index
    in label_maps["label_to_id"].items()
}

class_names = [
    label
    for label, index in sorted(
        label_to_id.items(),
        key=lambda item: item[1]
    )
]

print("Classes:", class_names)


# --------------------------------------------------
# Dataset class
# --------------------------------------------------

MAX_FRAMES = 169
FRAME_WIDTH = 1920
FRAME_HEIGHT = 1080


class PilotKeypointsDataset(Dataset):

    def __init__(
        self,
        folder,
        label_to_id,
        max_frames=169
    ):
        self.folder = Path(folder)
        self.files = sorted(
            self.folder.glob("*.json")
        )
        self.label_to_id = label_to_id
        self.max_frames = max_frames

    def __len__(self):
        return len(self.files)

    def interpolate_coordinates(self, values):
        values = np.asarray(
            values,
            dtype=np.float32
        )

        for landmark_index in range(
            values.shape[1]
        ):
            for coordinate_index in range(2):
                series = values[
                    :,
                    landmark_index,
                    coordinate_index
                ]

                valid = np.isfinite(series)

                if valid.any():
                    indices = np.arange(
                        len(series)
                    )

                    series[~valid] = np.interp(
                        indices[~valid],
                        indices[valid],
                        series[valid]
                    )
                else:
                    series[:] = 0.0

                values[
                    :,
                    landmark_index,
                    coordinate_index
                ] = series

        return values

    def __getitem__(self, index):
        file_path = self.files[index]

        with open(file_path, "r") as file:
            item = json.load(file)

        pose = np.stack(
            [item["pose_x"], item["pose_y"]],
            axis=-1
        )

        hand1 = np.stack(
            [item["hand1_x"], item["hand1_y"]],
            axis=-1
        )

        hand2 = np.stack(
            [item["hand2_x"], item["hand2_y"]],
            axis=-1
        )

        pose = self.interpolate_coordinates(pose)
        hand1 = self.interpolate_coordinates(hand1)
        hand2 = self.interpolate_coordinates(hand2)

        pose[:, :, 0] *= FRAME_WIDTH
        pose[:, :, 1] *= FRAME_HEIGHT

        hand1[:, :, 0] *= FRAME_WIDTH
        hand1[:, :, 1] *= FRAME_HEIGHT

        hand2[:, :, 0] *= FRAME_WIDTH
        hand2[:, :, 1] *= FRAME_HEIGHT

        features = np.concatenate(
            [pose, hand1, hand2],
            axis=1
        )

        features = features.reshape(
            features.shape[0],
            -1
        )

        if features.shape[0] > self.max_frames:
            features = features[:self.max_frames]

        elif features.shape[0] < self.max_frames:
            padding = np.zeros(
                (
                    self.max_frames
                    - features.shape[0],
                    features.shape[1]
                ),
                dtype=np.float32
            )

            features = np.concatenate(
                [features, padding],
                axis=0
            )

        label_name = item["label"]

        return {
            "data": torch.tensor(
                features,
                dtype=torch.float32
            ),
            "label": torch.tensor(
                self.label_to_id[label_name],
                dtype=torch.long
            ),
            "uid": item["uid"]
        }


# --------------------------------------------------
# Test loader
# --------------------------------------------------

keypoint_root = (
    research_dir /
    "pilot_keypoints"
)

test_dataset = PilotKeypointsDataset(
    keypoint_root / "pilot_test_keypoints",
    label_to_id,
    MAX_FRAMES
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)


# --------------------------------------------------
# INCLUDE Transformer architecture
# --------------------------------------------------

class TransformerConfig:

    def __init__(
        self,
        size="small",
        max_position_embeddings=256
    ):
        self.size = size
        self.input_size = 134
        self.max_position_embeddings = (
            max_position_embeddings
        )
        self.layer_norm_eps = 1e-12
        self.hidden_dropout_prob = 0.1

        if size == "small":
            self.hidden_size = 256
            self.num_attention_heads = 4
            self.num_hidden_layers = 2
        else:
            self.hidden_size = 512
            self.num_attention_heads = 8
            self.num_hidden_layers = 4

        self.model_config = transformers.BertConfig(
            hidden_size=self.hidden_size,
            num_attention_heads=(
                self.num_attention_heads
            ),
            num_hidden_layers=(
                self.num_hidden_layers
            ),
            max_position_embeddings=(
                self.max_position_embeddings
            )
        )

        self.model_config._attn_implementation = (
            "eager"
        )


class PositionEmbedding(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.position_embeddings = nn.Embedding(
            config.max_position_embeddings,
            config.hidden_size
        )

        self.LayerNorm = nn.LayerNorm(
            config.hidden_size,
            eps=config.layer_norm_eps
        )

        self.dropout = nn.Dropout(
            config.hidden_dropout_prob
        )

        self.register_buffer(
            "position_ids",
            torch.arange(
                config.max_position_embeddings
            ).expand((1, -1))
        )

    def forward(self, x):
        sequence_length = x.size(1)

        position_ids = self.position_ids[
            :,
            :sequence_length
        ]

        position_embeddings = (
            self.position_embeddings(
                position_ids
            )
        )

        x = x + position_embeddings
        x = self.LayerNorm(x)
        x = self.dropout(x)

        return x


class CompatibleTransformer(nn.Module):

    def __init__(
        self,
        config,
        number_of_classes
    ):
        super().__init__()

        self.l1 = nn.Linear(
            config.input_size,
            config.hidden_size
        )

        self.embedding = PositionEmbedding(
            config
        )

        self.layers = nn.ModuleList([
            transformers.BertLayer(
                config.model_config
            )
            for _ in range(
                config.num_hidden_layers
            )
        ])

        self.l2 = nn.Linear(
            config.hidden_size,
            number_of_classes
        )

    def forward(self, x):
        x = self.l1(x)
        x = self.embedding(x)

        for layer in self.layers:
            result = layer(x)

            if isinstance(result, (tuple, list)):
                x = result[0]
            else:
                x = result

        x = torch.max(x, dim=1).values

        x = F.dropout(
            x,
            p=0.2,
            training=self.training
        )

        return self.l2(x)


device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)
print("Test videos:", len(test_dataset))
print("Setup complete.")

In [ ]:
import gc
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from IPython.display import display as show_table


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


SEEDS = [42, 123, 2026]
NUMBER_OF_CLASSES = len(class_names)

keypoint_root = research_dir / "pilot_keypoints"

train_dataset = PilotKeypointsDataset(
    keypoint_root / "pilot_train_keypoints",
    label_to_id,
    MAX_FRAMES
)

val_dataset = PilotKeypointsDataset(
    keypoint_root / "pilot_val_keypoints",
    label_to_id,
    MAX_FRAMES
)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_train_loader(seed):
    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        train_dataset,
        batch_size=8,
        shuffle=True,
        num_workers=0,
        generator=generator
    )


val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)


def create_model():
    config = TransformerConfig(
        size="small",
        max_position_embeddings=256
    )

    return CompatibleTransformer(
        config=config,
        number_of_classes=NUMBER_OF_CLASSES
    ).to(device)


def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None

    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for batch in loader:
        inputs = batch["data"].to(device)
        labels = batch["label"].to(device)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            logits = model(inputs)
            loss = criterion(logits, labels)

            if training:
                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0
                )

                optimizer.step()

        predictions = torch.argmax(logits, dim=1)

        total_loss += loss.item() * labels.size(0)
        total_correct += (
            predictions == labels
        ).sum().item()
        total_samples += labels.size(0)

    return (
        total_loss / total_samples,
        total_correct / total_samples
    )


def fit_model(
    model,
    train_loader,
    learning_rate,
    maximum_epochs,
    patience,
    experiment_name,
    seed
):
    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        [
            parameter
            for parameter in model.parameters()
            if parameter.requires_grad
        ],
        lr=learning_rate,
        weight_decay=1e-4
    )

    best_accuracy = -1.0
    best_loss = float("inf")
    best_state = None
    best_epoch = 0
    waiting = 0
    history = []

    for epoch in range(maximum_epochs):
        train_loss, train_accuracy = run_epoch(
            model,
            train_loader,
            criterion,
            optimizer
        )

        val_loss, val_accuracy = run_epoch(
            model,
            val_loader,
            criterion
        )

        history.append({
            "Model": experiment_name,
            "Seed": seed,
            "Epoch": epoch + 1,
            "Train Loss": train_loss,
            "Train Accuracy": train_accuracy,
            "Validation Loss": val_loss,
            "Validation Accuracy": val_accuracy
        })

        print(
            f"{experiment_name} | seed {seed} | "
            f"epoch {epoch + 1}/{maximum_epochs} | "
            f"train {train_accuracy:.3f} | "
            f"validation {val_accuracy:.3f}"
        )

        improved = (
            val_accuracy > best_accuracy
            or (
                val_accuracy == best_accuracy
                and val_loss < best_loss
            )
        )

        if improved:
            best_accuracy = val_accuracy
            best_loss = val_loss
            best_epoch = epoch + 1
            waiting = 0

            best_state = {
                key: value.detach().cpu().clone()
                for key, value
                in model.state_dict().items()
            }

        else:
            waiting += 1

            if waiting >= patience:
                break

    model.load_state_dict(best_state)

    return model, best_epoch, best_accuracy, history


def evaluate_test(model, model_name, seed, best_epoch):
    model.eval()

    true_labels = []
    predictions = []

    with torch.no_grad():
        for batch in test_loader:
            inputs = batch["data"].to(device)
            labels = batch["label"].to(device)

            logits = model(inputs)

            predicted = torch.argmax(
                logits,
                dim=1
            )

            true_labels.extend(
                labels.cpu().tolist()
            )

            predictions.extend(
                predicted.cpu().tolist()
            )

    correct = sum(
        true == predicted
        for true, predicted
        in zip(true_labels, predictions)
    )

    return {
        "Model": model_name,
        "Seed": seed,
        "Best Epoch": best_epoch,
        "Correct": correct,
        "Test Videos": len(true_labels),
        "Accuracy": accuracy_score(
            true_labels,
            predictions
        ),
        "Macro Precision": precision_score(
            true_labels,
            predictions,
            average="macro",
            zero_division=0
        ),
        "Macro Recall": recall_score(
            true_labels,
            predictions,
            average="macro",
            zero_division=0
        ),
        "Macro F1": f1_score(
            true_labels,
            predictions,
            average="macro",
            zero_division=0
        )
    }

In [ ]:
# The backbone in this checkpoint remained frozen,
# so it contains the original pretrained INCLUDE weights.

pretrained_checkpoint = torch.load(
    research_dir / "pilot_head_best.pth",
    map_location="cpu",
    weights_only=False
)

pretrained_backbone = {
    key: value
    for key, value
    in pretrained_checkpoint["model"].items()
    if not key.startswith("l2.")
}

checkpoint_folder = (
    research_dir /
    "multiseed_checkpoints"
)

checkpoint_folder.mkdir(
    parents=True,
    exist_ok=True
)

all_results = []
all_history = []


for seed in SEEDS:
    print("\n==============================")
    print("SEED:", seed)
    print("==============================")

    train_loader = make_train_loader(seed)

    # ----------------------------------------------
    # A. Transformer trained from scratch
    # ----------------------------------------------

    set_seed(seed)

    scratch_model = create_model()

    scratch_model, scratch_epoch, _, history = fit_model(
        model=scratch_model,
        train_loader=train_loader,
        learning_rate=1e-4,
        maximum_epochs=50,
        patience=7,
        experiment_name="Scratch",
        seed=seed
    )

    all_history.extend(history)

    all_results.append(
        evaluate_test(
            scratch_model,
            "Scratch",
            seed,
            scratch_epoch
        )
    )

    torch.save(
        {"model": scratch_model.state_dict()},
        checkpoint_folder /
        f"scratch_seed_{seed}.pth"
    )

    del scratch_model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ----------------------------------------------
    # B. Pretrained frozen backbone
    # ----------------------------------------------

    set_seed(seed)

    frozen_model = create_model()

    missing, unexpected = frozen_model.load_state_dict(
        pretrained_backbone,
        strict=False
    )

    for parameter in frozen_model.parameters():
        parameter.requires_grad = False

    for parameter in frozen_model.l2.parameters():
        parameter.requires_grad = True

    frozen_model, frozen_epoch, _, history = fit_model(
        model=frozen_model,
        train_loader=train_loader,
        learning_rate=1e-3,
        maximum_epochs=20,
        patience=5,
        experiment_name="Pretrained Frozen",
        seed=seed
    )

    all_history.extend(history)

    all_results.append(
        evaluate_test(
            frozen_model,
            "Pretrained Frozen",
            seed,
            frozen_epoch
        )
    )

    torch.save(
        {"model": frozen_model.state_dict()},
        checkpoint_folder /
        f"frozen_seed_{seed}.pth"
    )

    # ----------------------------------------------
    # C. Full fine-tuning
    # Begins from the validation-selected frozen model
    # ----------------------------------------------

    for parameter in frozen_model.parameters():
        parameter.requires_grad = True

    set_seed(seed)

    full_model, full_epoch, _, history = fit_model(
        model=frozen_model,
        train_loader=train_loader,
        learning_rate=1e-5,
        maximum_epochs=20,
        patience=5,
        experiment_name="Full Fine-tuning",
        seed=seed
    )

    all_history.extend(history)

    all_results.append(
        evaluate_test(
            full_model,
            "Full Fine-tuning",
            seed,
            full_epoch
        )
    )

    torch.save(
        {"model": full_model.state_dict()},
        checkpoint_folder /
        f"full_seed_{seed}.pth"
    )

    del full_model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


results_df = pd.DataFrame(all_results)
history_df = pd.DataFrame(all_history)

results_path = (
    research_dir /
    "pilot_multiseed_results.csv"
)

history_path = (
    research_dir /
    "pilot_multiseed_training_history.csv"
)

results_df.to_csv(
    results_path,
    index=False
)

history_df.to_csv(
    history_path,
    index=False
)

print("\nMULTI-SEED RESULTS")
show_table(results_df.round(4))

print("Saved results:", results_path)
print("Saved history:", history_path)`

In [ ]:
fair_scratch_results = []
fair_scratch_history = []

for seed in SEEDS:
    print("\n==============================")
    print("FAIR SCRATCH SEED:", seed)
    print("==============================")

    set_seed(seed)

    train_loader = make_train_loader(seed)
    scratch_model = create_model()

    # Patience=60 prevents premature stopping.
    # The best checkpoint is still selected using validation only.
    scratch_model, best_epoch, _, history = fit_model(
        model=scratch_model,
        train_loader=train_loader,
        learning_rate=1e-4,
        maximum_epochs=60,
        patience=60,
        experiment_name="Scratch",
        seed=seed
    )

    fair_scratch_history.extend(history)

    result = evaluate_test(
        scratch_model,
        "Scratch",
        seed,
        best_epoch
    )

    fair_scratch_results.append(result)

    torch.save(
        {
            "model": scratch_model.state_dict(),
            "seed": seed,
            "best_epoch": best_epoch
        },
        checkpoint_folder /
        f"scratch_fair_seed_{seed}.pth"
    )

    print(
        "Final:",
        result["Correct"],
        "/",
        result["Test Videos"],
        "accuracy:",
        round(result["Accuracy"], 4)
    )

    del scratch_model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# Keep existing pretrained results
pretrained_results = results_df[
    results_df["Model"] != "Scratch"
].copy()

# Add corrected scratch results
results_df = pd.concat(
    [
        pretrained_results,
        pd.DataFrame(fair_scratch_results)
    ],
    ignore_index=True
)

results_df = results_df.sort_values(
    ["Seed", "Model"]
).reset_index(drop=True)

# Replace old scratch training history
pretrained_history = history_df[
    history_df["Model"] != "Scratch"
].copy()

history_df = pd.concat(
    [
        pretrained_history,
        pd.DataFrame(fair_scratch_history)
    ],
    ignore_index=True
)

results_df.to_csv(
    research_dir /
    "pilot_multiseed_results.csv",
    index=False
)

history_df.to_csv(
    research_dir /
    "pilot_multiseed_training_history.csv",
    index=False
)

show_table(results_df.round(4))

In [ ]:
summary = (
    results_df
    .groupby("Model")[
        [
            "Accuracy",
            "Macro Precision",
            "Macro Recall",
            "Macro F1"
        ]
    ]
    .agg(["mean", "std"])
)

summary.columns = [
    f"{metric} {statistic}"
    for metric, statistic
    in summary.columns
]

summary = summary.reset_index()

summary_path = (
    research_dir /
    "pilot_multiseed_summary.csv"
)

summary.to_csv(
    summary_path,
    index=False
)

show_table(summary.round(4))

print("Saved corrected results and summary.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

model_order = [
    "Scratch",
    "Pretrained Frozen",
    "Full Fine-tuning"
]

ordered_summary = (
    summary
    .set_index("Model")
    .loc[model_order]
    .reset_index()
)

paper_table = pd.DataFrame({
    "Model": [
        "Scratch Transformer",
        "Pretrained + Frozen Backbone",
        "Pretrained + Full Fine-tuning"
    ],
    "Accuracy": [
        f"{mean * 100:.2f}% ± {std * 100:.2f}%"
        for mean, std in zip(
            ordered_summary["Accuracy mean"],
            ordered_summary["Accuracy std"]
        )
    ],
    "Macro-F1": [
        f"{mean * 100:.2f}% ± {std * 100:.2f}%"
        for mean, std in zip(
            ordered_summary["Macro F1 mean"],
            ordered_summary["Macro F1 std"]
        )
    ]
})

paper_table_path = (
    research_dir /
    "pilot_paper_results_table.csv"
)

paper_table.to_csv(
    paper_table_path,
    index=False
)

show_table(paper_table)

# Graph with standard-deviation error bars
x = np.arange(len(model_order))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))

ax.bar(
    x - width / 2,
    ordered_summary["Accuracy mean"],
    width,
    yerr=ordered_summary["Accuracy std"],
    capsize=5,
    color="#333333",
    label="Accuracy"
)

ax.bar(
    x + width / 2,
    ordered_summary["Macro F1 mean"],
    width,
    yerr=ordered_summary["Macro F1 std"],
    capsize=5,
    color="#999999",
    label="Macro-F1"
)

ax.set_xticks(x)

ax.set_xticklabels(
    [
        "Scratch",
        "Pretrained\nFrozen",
        "Full\nFine-tuning"
    ]
)

ax.set_ylim(0.80, 1.01)
ax.set_ylabel("Score")
ax.set_title("Three-Seed Model Comparison")
ax.legend()
ax.grid(axis="y", alpha=0.25)

fig.tight_layout()

figure_path = (
    research_dir /
    "pilot_multiseed_comparison.png"
)

fig.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Paper table:", paper_table_path)
print("Paper figure:", figure_path)

In [ ]:
import json
import torch

final_checkpoint_path = (
    research_dir /
    "multiseed_checkpoints/frozen_seed_2026.pth"
)

final_checkpoint = torch.load(
    final_checkpoint_path,
    map_location=device,
    weights_only=False
)

final_model = create_model()

final_model.load_state_dict(
    final_checkpoint["model"],
    strict=True
)

final_model.eval()

# Change this from 0 to 33 to inspect another test video
sample_index = 0

sample = test_dataset[sample_index]

input_tensor = (
    sample["data"]
    .unsqueeze(0)
    .to(device)
)

with torch.no_grad():
    logits = final_model(input_tensor)

    probabilities = torch.softmax(
        logits,
        dim=1
    )[0]

predicted_id = int(
    torch.argmax(probabilities).item()
)

true_id = int(sample["label"].item())

predicted_label = class_names[predicted_id]
true_label = class_names[true_id]
confidence = float(
    probabilities[predicted_id].item()
)

top_probabilities, top_indices = torch.topk(
    probabilities,
    k=3
)

top_three = []

for probability, index in zip(
    top_probabilities.cpu().tolist(),
    top_indices.cpu().tolist()
):
    top_three.append({
        "label": class_names[index],
        "confidence": probability
    })

print("UID:", sample["uid"])
print("True sign:", true_label)
print("Predicted sign:", predicted_label)
print("Confidence:", f"{confidence * 100:.2f}%")
print("Correct:", predicted_id == true_id)

print("\nTop-three predictions:")

for rank, result in enumerate(top_three, start=1):
    print(
        f"{rank}. {result['label']}: "
        f"{result['confidence'] * 100:.2f}%"
    )

demo_result = {
    "uid": sample["uid"],
    "true_label": true_label,
    "predicted_label": predicted_label,
    "confidence": confidence,
    "correct": predicted_id == true_id,
    "top_three": top_three
}

demo_json_path = (
    research_dir /
    "pilot_demo_prediction.json"
)

with open(demo_json_path, "w") as file:
    json.dump(
        demo_result,
        file,
        indent=2
    )

print("\nSaved:", demo_json_path)

In [ ]:
import cv2
import matplotlib.pyplot as plt

uid = sample["uid"]

prefix = true_label + "_"

if uid.startswith(prefix):
    video_stem = uid[len(prefix):]
else:
    video_stem = uid

video_folder = (
    research_dir /
    "pilot_videos" /
    true_label
)

video_matches = [
    path
    for path in video_folder.iterdir()
    if path.is_file()
    and path.stem == video_stem
]

if not video_matches:
    video_matches = [
        path
        for path in (
            research_dir /
            "pilot_videos"
        ).rglob("*")
        if path.is_file()
        and path.stem == video_stem
    ]

assert video_matches, (
    f"Original video not found for {uid}"
)

video_path = video_matches[0]

capture = cv2.VideoCapture(
    str(video_path)
)

frame_count = int(
    capture.get(
        cv2.CAP_PROP_FRAME_COUNT
    )
)

capture.set(
    cv2.CAP_PROP_POS_FRAMES,
    max(frame_count // 2, 0)
)

success, frame = capture.read()
capture.release()

assert success, "Could not read video frame."

frame = cv2.cvtColor(
    frame,
    cv2.COLOR_BGR2RGB
)

top_labels = [
    result["label"]
    for result in reversed(top_three)
]

top_scores = [
    result["confidence"]
    for result in reversed(top_three)
]

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5)
)

axes[0].imshow(frame)
axes[0].axis("off")
axes[0].set_title(
    f"True sign: {true_label}"
)

axes[1].barh(
    top_labels,
    top_scores,
    color="#555555"
)

axes[1].set_xlim(0, 1)
axes[1].set_xlabel("Model confidence")
axes[1].set_title(
    f"Prediction: {predicted_label}"
)

for index, score in enumerate(top_scores):
    axes[1].text(
        score + 0.01,
        index,
        f"{score * 100:.1f}%",
        va="center"
    )

fig.suptitle(
    "Dhwani ISL Recognition — Example Output",
    fontsize=14
)

fig.tight_layout()

demo_figure_path = (
    research_dir /
    "pilot_demo_prediction.png"
)

fig.savefig(
    demo_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Video:", video_path)
print("Saved figure:", demo_figure_path)

In [ ]:
!pip install -q onnx onnxruntime

In [ ]:
import copy
import json
import torch

onnx_path = (
    research_dir /
    "dhwani_include_transformer.onnx"
)

labels_path = (
    research_dir /
    "dhwani_labels.txt"
)

# Copy the model to CPU for export
export_model = copy.deepcopy(
    final_model
).cpu().eval()

example_input = (
    sample["data"]
    .unsqueeze(0)
    .cpu()
)

print("Example input:", example_input.shape)

torch.onnx.export(
    export_model,
    example_input,
    str(onnx_path),
    input_names=["keypoints"],
    output_names=["logits"],
    opset_version=17,
    dynamo=False
)

with open(labels_path, "w") as file:
    for label in class_names:
        file.write(label + "\n")

print("ONNX model:", onnx_path)
print("Labels:", labels_path)
print(
    "Model size:",
    round(
        onnx_path.stat().st_size /
        (1024 * 1024),
        2
    ),
    "MB"
)

In [ ]:
import json
import numpy as np
import onnx
import onnxruntime as ort
import torch

# Validate ONNX structure
onnx_model = onnx.load(
    str(onnx_path)
)

onnx.checker.check_model(
    onnx_model
)

print("ONNX structure is valid.")

# Run ONNX inference
session = ort.InferenceSession(
    str(onnx_path),
    providers=["CPUExecutionProvider"]
)

onnx_input = (
    example_input
    .numpy()
    .astype(np.float32)
)

onnx_logits = session.run(
    ["logits"],
    {"keypoints": onnx_input}
)[0]

# Run PyTorch inference
with torch.no_grad():
    pytorch_logits = (
        export_model(example_input)
        .numpy()
    )

maximum_difference = float(
    np.max(
        np.abs(
            pytorch_logits -
            onnx_logits
        )
    )
)

pytorch_prediction = int(
    np.argmax(pytorch_logits, axis=1)[0]
)

onnx_prediction = int(
    np.argmax(onnx_logits, axis=1)[0]
)

def softmax(values):
    shifted = values - np.max(
        values,
        axis=1,
        keepdims=True
    )

    exponentials = np.exp(shifted)

    return exponentials / np.sum(
        exponentials,
        axis=1,
        keepdims=True
    )

onnx_probabilities = softmax(
    onnx_logits
)

onnx_confidence = float(
    onnx_probabilities[
        0,
        onnx_prediction
    ]
)

print("PyTorch prediction:", class_names[pytorch_prediction])
print("ONNX prediction:", class_names[onnx_prediction])
print("ONNX confidence:", f"{onnx_confidence * 100:.2f}%")
print("Maximum output difference:", maximum_difference)
print(
    "Predictions match:",
    pytorch_prediction == onnx_prediction
)

verification = {
    "input_shape": list(onnx_input.shape),
    "pytorch_prediction": class_names[pytorch_prediction],
    "onnx_prediction": class_names[onnx_prediction],
    "onnx_confidence": onnx_confidence,
    "maximum_output_difference": maximum_difference,
    "predictions_match": (
        pytorch_prediction ==
        onnx_prediction
    )
}

verification_path = (
    research_dir /
    "onnx_verification.json"
)

with open(verification_path, "w") as file:
    json.dump(
        verification,
        file,
        indent=2
    )

print("Verification saved:", verification_path)

In [ ]:
from pathlib import Path
import os
import onnx
import onnxruntime as ort

from onnxruntime.quantization import (
    quantize_dynamic,
    QuantType
)

research_dir = Path(
    "/content/drive/MyDrive/DhwaniResearch"
)

fp32_model_path = (
    research_dir /
    "dhwani_include_transformer.onnx"
)

int8_model_path = (
    research_dir /
    "dhwani_include_transformer_int8.onnx"
)

quantize_dynamic(
    model_input=str(fp32_model_path),
    model_output=str(int8_model_path),
    weight_type=QuantType.QInt8
)

# Validate the generated ONNX file
int8_model = onnx.load(str(int8_model_path))
onnx.checker.check_model(int8_model)

fp32_size_mb = os.path.getsize(fp32_model_path) / (1024 ** 2)
int8_size_mb = os.path.getsize(int8_model_path) / (1024 ** 2)

reduction_percent = (
    (fp32_size_mb - int8_size_mb)
    / fp32_size_mb
) * 100

print("INT8 ONNX structure is valid.")
print("FP32 size:", round(fp32_size_mb, 2), "MB")
print("INT8 size:", round(int8_size_mb, 2), "MB")
print(
    "Size reduction:",
    round(reduction_percent, 2),
    "%"
)
print("Saved:", int8_model_path)

In [ ]:
import json
import numpy as np
import pandas as pd
import onnxruntime as ort

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

# Create ONNX Runtime sessions
fp32_session = ort.InferenceSession(
    str(fp32_model_path),
    providers=["CPUExecutionProvider"]
)

int8_session = ort.InferenceSession(
    str(int8_model_path),
    providers=["CPUExecutionProvider"]
)

fp32_input_name = fp32_session.get_inputs()[0].name
int8_input_name = int8_session.get_inputs()[0].name

print("FP32 input:", fp32_input_name)
print("INT8 input:", int8_input_name)

true_labels = []
fp32_predictions = []
int8_predictions = []
all_uids = []

# The exported graph uses batch size 1,
# so every test sample is evaluated separately.
for batch in test_loader:
    batch_data = batch["data"].cpu().numpy()
    batch_labels = batch["label"].cpu().numpy()
    batch_uids = batch["uid"]

    for index in range(len(batch_labels)):
        sample = batch_data[index:index + 1].astype(
            np.float32
        )

        fp32_logits = fp32_session.run(
            None,
            {fp32_input_name: sample}
        )[0]

        int8_logits = int8_session.run(
            None,
            {int8_input_name: sample}
        )[0]

        fp32_prediction = int(
            np.argmax(fp32_logits, axis=1)[0]
        )

        int8_prediction = int(
            np.argmax(int8_logits, axis=1)[0]
        )

        true_labels.append(int(batch_labels[index]))
        fp32_predictions.append(fp32_prediction)
        int8_predictions.append(int8_prediction)
        all_uids.append(batch_uids[index])

# Calculate FP32 metrics
fp32_accuracy = accuracy_score(
    true_labels,
    fp32_predictions
)

fp32_precision = precision_score(
    true_labels,
    fp32_predictions,
    average="macro",
    zero_division=0
)

fp32_recall = recall_score(
    true_labels,
    fp32_predictions,
    average="macro",
    zero_division=0
)

fp32_f1 = f1_score(
    true_labels,
    fp32_predictions,
    average="macro",
    zero_division=0
)

# Calculate INT8 metrics
int8_accuracy = accuracy_score(
    true_labels,
    int8_predictions
)

int8_precision = precision_score(
    true_labels,
    int8_predictions,
    average="macro",
    zero_division=0
)

int8_recall = recall_score(
    true_labels,
    int8_predictions,
    average="macro",
    zero_division=0
)

int8_f1 = f1_score(
    true_labels,
    int8_predictions,
    average="macro",
    zero_division=0
)

# Agreement between the two ONNX models
agreement = np.mean(
    np.array(fp32_predictions)
    == np.array(int8_predictions)
)

fp32_correct = int(
    np.sum(
        np.array(true_labels)
        == np.array(fp32_predictions)
    )
)

int8_correct = int(
    np.sum(
        np.array(true_labels)
        == np.array(int8_predictions)
    )
)

print("\nFP32 ONNX RESULTS")
print("Correct:", fp32_correct, "/", len(true_labels))
print("Accuracy:", round(fp32_accuracy, 4))
print("Macro precision:", round(fp32_precision, 4))
print("Macro recall:", round(fp32_recall, 4))
print("Macro F1:", round(fp32_f1, 4))

print("\nINT8 ONNX RESULTS")
print("Correct:", int8_correct, "/", len(true_labels))
print("Accuracy:", round(int8_accuracy, 4))
print("Macro precision:", round(int8_precision, 4))
print("Macro recall:", round(int8_recall, 4))
print("Macro F1:", round(int8_f1, 4))

print(
    "\nFP32/INT8 agreement:",
    f"{agreement * 100:.2f}%"
)

print(
    "Accuracy difference:",
    round(int8_accuracy - fp32_accuracy, 4)
)

In [ ]:
quantization_results = {
    "fp32_model": fp32_model_path.name,
    "int8_model": int8_model_path.name,
    "test_videos": len(true_labels),

    "fp32_size_mb": round(fp32_size_mb, 4),
    "int8_size_mb": round(int8_size_mb, 4),
    "size_reduction_percent": round(
        reduction_percent,
        4
    ),

    "fp32_correct": fp32_correct,
    "fp32_accuracy": float(fp32_accuracy),
    "fp32_macro_precision": float(fp32_precision),
    "fp32_macro_recall": float(fp32_recall),
    "fp32_macro_f1": float(fp32_f1),

    "int8_correct": int8_correct,
    "int8_accuracy": float(int8_accuracy),
    "int8_macro_precision": float(int8_precision),
    "int8_macro_recall": float(int8_recall),
    "int8_macro_f1": float(int8_f1),

    "prediction_agreement": float(agreement),
    "accuracy_difference": float(
        int8_accuracy - fp32_accuracy
    )
}

results_path = (
    research_dir /
    "onnx_quantization_results.json"
)

with open(results_path, "w") as file:
    json.dump(
        quantization_results,
        file,
        indent=2
    )

comparison_table = pd.DataFrame({
    "uid": all_uids,
    "true_label": [
        class_names[index]
        for index in true_labels
    ],
    "fp32_prediction": [
        class_names[index]
        for index in fp32_predictions
    ],
    "int8_prediction": [
        class_names[index]
        for index in int8_predictions
    ],
    "fp32_correct": [
        true == prediction
        for true, prediction in zip(
            true_labels,
            fp32_predictions
        )
    ],
    "int8_correct": [
        true == prediction
        for true, prediction in zip(
            true_labels,
            int8_predictions
        )
    ],
    "models_agree": [
        fp32 == int8
        for fp32, int8 in zip(
            fp32_predictions,
            int8_predictions
        )
    ]
})

comparison_path = (
    research_dir /
    "onnx_fp32_int8_predictions.csv"
)

comparison_table.to_csv(
    comparison_path,
    index=False
)

print("Results saved:", results_path)
print("Predictions saved:", comparison_path)

print("\nCases where FP32 and INT8 disagree:")
from IPython.display import display as show_table

show_table(
    comparison_table[
        comparison_table["models_agree"] == False
    ]
)